# LightGBM Training — Safe Lag Features

Retrain LightGBM với **safe lag feature set** (từ `splits_safe_lag/`):
- **Xóa**: lag_1, lag_2, lag_3, lag_7, rolling_mean_7, rolling_mean_14
- **Thêm**: lag_16, rolling_mean_30
- **GPU**: `device='gpu'`
- **Params**: Optuna best params từ lần tuning trước (không tune lại)

## 1. Setup & Load Data

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SPLITS_SAFE   = PROCESSED_DIR / 'splits_safe_lag'

assert SPLITS_SAFE.exists(), (
    f'splits_safe_lag không tồn tại: {SPLITS_SAFE}\n'
    f'Hãy chạy feature_safe_lag_v2.ipynb trước!'
)
print(f'SPLITS_SAFE: {SPLITS_SAFE}')
print(f'MODELS_DIR : {MODELS_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import joblib
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error

print('Loading splits từ splits_safe_lag/...')
X_train    = pd.read_csv(SPLITS_SAFE / 'train_features.csv')
y_train    = pd.read_csv(SPLITS_SAFE / 'train_target.csv')
X_val      = pd.read_csv(SPLITS_SAFE / 'val_features.csv')
y_val      = pd.read_csv(SPLITS_SAFE / 'val_target.csv')
X_test     = pd.read_csv(SPLITS_SAFE / 'test_features.csv')
y_test     = pd.read_csv(SPLITS_SAFE / 'test_target.csv')
y_val_orig = pd.read_csv(SPLITS_SAFE / 'val_target_original.csv')

print(f'X_train    : {X_train.shape}')
print(f'X_val      : {X_val.shape}')
print(f'X_test     : {X_test.shape}')
print(f'\nNew features present:')
for col in ['lag_16', 'rolling_mean_30']:
    in_df = col in X_train.columns
    print(f'  {col:<20}: {"YES ✓" if in_df else "MISSING ✗"}')
print(f'\nRemoved features absent:')
for col in ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_mean_7', 'rolling_mean_14']:
    absent = col not in X_train.columns
    print(f'  {col:<20}: {"absent ✓" if absent else "STILL PRESENT ✗"}')

## 1b. Encode Categorical Features

In [ ]:
object_cols    = X_train.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}

for col in object_cols:
    le    = LabelEncoder()
    parts = [X_train[col], X_val[col]]
    if col in X_test.columns:
        parts.append(X_test[col])
    combined = pd.concat(parts, ignore_index=True).astype(str)
    le.fit(combined)

    X_train[col] = le.transform(X_train[col].astype(str)).astype(np.int32)
    X_val[col]   = le.transform(X_val[col].astype(str)).astype(np.int32)
    if col in X_test.columns:
        X_test[col] = le.transform(X_test[col].astype(str)).astype(np.int32)

    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} labels')

remaining = X_train.select_dtypes(include=['object']).columns.tolist()
if remaining:
    raise ValueError(f'Still has object columns: {remaining}')

print(f'\nLabelEncoded {len(object_cols)} cols | '
      f'X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}')

## 2. Metrics Helper

In [ ]:
y_test_orig = pd.DataFrame({'sales': np.expm1(y_test['sales_log'])})

def evaluate_metrics(y_true_log, y_pred_log, y_true_orig, label=''):
    y_pred_orig = np.clip(np.expm1(y_pred_log), 0, None)
    y_true_orig = np.clip(y_true_orig, 0, None)

    rmsle    = np.sqrt(mean_squared_log_error(y_true_orig, y_pred_orig))
    rmse     = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mae      = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    if label:
        print(f'\nLightGBM — {label}:')
        print(f'  RMSLE              : {rmsle:.6f}')
        print(f'  RMSE (sales units) : {rmse:.4f}')
        print(f'  MAE  (sales units) : {mae:.4f}')
        print(f'  RMSE (log scale)   : {rmse_log:.6f}')

    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'RMSE_log': rmse_log}

## 3. Load Optuna Best Params & Retrain với GPU

In [ ]:
params_file = MODELS_DIR / 'lightgbm_optuna_best_params.json'
assert params_file.exists(), f'Optuna params không tìm thấy: {params_file}'

with open(params_file, 'r') as f:
    best_optuna_params = json.load(f)

print('Optuna best params (từ LightGBM_training_v2_out.ipynb):')
for k, v in best_optuna_params.items():
    print(f'  {k:<20}: {v}')

In [ ]:
# ── Combine train + val cho final retraining ────────────────────────────────
X_tv = pd.concat([X_train, X_val], ignore_index=True)
y_tv = pd.concat([y_train['sales_log'], y_val['sales_log']], ignore_index=True)

print(f'Train+Val: {X_tv.shape}')

# ── Best params với GPU ─────────────────────────────────────────────────────
# Note: LightGBM dùng device='gpu', KHÔNG dùng tree_method (đó là XGBoost param)
best_params = {
    **best_optuna_params,
    'n_estimators': 1000,
    'objective':    'regression',
    'random_state': 42,
    'n_jobs':       -1,
    'verbose':      -1,
    'device':       'gpu',       # GPU acceleration
}

print('\nFinal training params:')
for k, v in best_params.items():
    print(f'  {k:<20}: {v}')

model_file = MODELS_DIR / 'lightgbm_safe_lag_model.pkl'

if os.path.exists(model_file):
    print(f'\nFound {model_file.name}. Loading saved model...')
    best_model = joblib.load(model_file)
else:
    print('\nTraining LightGBM với GPU + safe lag features...')
    best_model = lgb.LGBMRegressor(**best_params)
    best_model.fit(X_tv, y_tv)
    os.makedirs(model_file.parent, exist_ok=True)
    joblib.dump(best_model, model_file)
    print(f'\nSaved: {model_file}')

## 4. Evaluation & So sánh với Model Cũ

In [ ]:
val_metrics  = evaluate_metrics(
    y_val['sales_log'],
    best_model.predict(X_val),
    y_val_orig['sales'],
    'Val  (safe lag)'
)

test_metrics = evaluate_metrics(
    y_test['sales_log'],
    best_model.predict(X_test),
    y_test_orig['sales'],
    'Test (safe lag) [Aug 01–15]'
)

print('\n' + '='*65)
print('SO SÁNH: LightGBM safe_lag vs LightGBM cũ (toxic lags)')
print('='*65)
OLD_VAL_RMSLE  = 0.353654
OLD_TEST_RMSLE = 0.373215
print(f'  Val  RMSLE — cũ : {OLD_VAL_RMSLE:.6f}')
print(f'  Val  RMSLE — mới: {val_metrics["RMSLE"]:.6f}  (delta: {val_metrics["RMSLE"] - OLD_VAL_RMSLE:+.6f})')
print(f'  Test RMSLE — cũ : {OLD_TEST_RMSLE:.6f}')
print(f'  Test RMSLE — mới: {test_metrics["RMSLE"]:.6f}  (delta: {test_metrics["RMSLE"] - OLD_TEST_RMSLE:+.6f})')
print('='*65)
print('Note: Val/Test metrics đo trên dữ liệu cũ (Aug 01-15).')
print('Lợi ích chính là zero NaN cho Kaggle test (Aug 16-31).')

## 5. Save Model

In [ ]:
# Lưu copy vào 08_forecasting/ để forecasting notebook dùng
save_dir = NOTEBOOKS_DIR / '08_forecasting'
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, save_dir / 'lgb_safe_lag_model.pkl')
print(f'Saved copy: {save_dir / "lgb_safe_lag_model.pkl"}')
print(f'Main model: {MODELS_DIR / "lightgbm_safe_lag_model.pkl"}')
print(f'\nModel n_features_: {best_model.n_features_}')